<a href="https://colab.research.google.com/github/ShashankSatishkumar/Lasers/blob/main/Ytterbium_fiberampsimul.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
h=6.626e-34
c=3.0e8
# p_w and s_w are pump and signal wavelengths
# p_f and s_f are pump and signal frequencies
p_w=976e-9
s_w=1030e-9
p_f=c/p_w
s_f=c/s_w
#acs is absorption cross section and ecs is emission cross section p=pump,s=signal
acs_p=2.5e-24
ecs_p=2.5e-24
acs_s=0.06e-24
ecs_s=0.6e-24
r_core=3e-6
a_core=np.pi*r_core**2
n_total=2.0e25
#tau=tsp=1/A21 (A21=spontaneous emission enstein coeffecient)
tau=1.0e-3
#fiber length
L=10.0
#o_p and o_s are overlap factors
o_p=0.05
o_s=0.8
#l is background loss
l_p=0
l_s=0.01
#initial input power
Po_p=0.6
Po_s=1.0e-3
def eq_solve(z,y):
# y[0] is pump power at z
# y[1] is signal power at z
  p_p=max(0.0,y[0])
  p_s=max(0.0,y[1])
  W_12=(o_p*acs_p*p_p)/(h*p_f*a_core)+(o_s*acs_s*p_s)/(h*s_f*a_core)
  W_21=(o_p*ecs_p*p_p)/(h*p_f*a_core)+(o_s*ecs_s*p_s)/(h*s_f*a_core)
#Steady state populations in both manifold levels
  n_2=n_total*(W_12)/(W_12+W_21+1/tau)
  n_1=n_total-n_2
# derivatives of power along dz
  pDp_dz=o_p*(ecs_p*n_2-acs_p*n_1)*p_p-l_p*p_p
  sDp_dz=o_s*(ecs_s*n_2-acs_s*n_1)*p_s-l_s*p_s
  return [pDp_dz,sDp_dz]
z_span=(0, L)
z_eval=np.linspace(0, L, 500)
init_p=[Po_p,Po_s]
sol= solve_ivp(eq_solve, z_span, init_p, t_eval=z_eval, method='RK45')
pos=sol.t
pump_prof=sol.y[0]
signal_prof=sol.y[1]
p_outp=pump_prof[-1]
s_outp=signal_prof[-1]
dbgain=10*np.log10(s_outp/Po_s)
print("results:")
print(f"signal output power : {s_outp:.5f} W")
print(f"pump power at end: {p_outp:.5f} W")
plt.figure(figsize=(9, 5))
plt.plot(pos, pump_prof, 'r-', linewidth=2, label='Pump Power (976 nm)')
plt.plot(pos, signal_prof, 'b-', linewidth=2, label='Signal Power (1030 nm)')
plt.xlabel('Fiber Position $z$ (meters)', fontsize=11)
plt.ylabel('Optical Power (Watts)', fontsize=11)
plt.title('Power Evolution along the Ytterbium-Doped Fiber', fontsize=12, fontweight='bold')
plt.legend(loc='best')
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()
